In [2]:
import os
import sys
from pathlib import Path
import asyncio
import json

project_root = Path(os.getcwd())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from schemas.models import ISO15926Model
from agents.super_agent import run_pipeline
from state.session_store import session_store
from utils.config import get_settings

cfg = get_settings()
print("Settings loaded. Redis URL:", cfg.redis_url)

NameError: name '__file__' is not defined

In [ ]:
# Load existing ISO 15926 model from JSON (engineering_model_iso15926.json)

model_path = Path("data/engineering_model_iso15926.json")
with model_path.open("r", encoding="utf-8") as f:
    data = json.load(f)

iso_model = ISO15926Model.model_validate(data)
print("Entities:", len(iso_model.entities), "Requirements:", len(iso_model.get_requirements()))

In [ ]:
from agents.research_agent import run_research_agent

async def test_research_only():
    session_id = "notebook-session-json"

    async def dummy_broadcast(sid: str, payload: dict):
        step = payload.get("step", "")
        agent = payload.get("agent", "")
        status = payload.get("status", "")
        if agent == "research" and (step.startswith("researching_") or step.startswith("done_") or step == "completed"):
            print(f"[{agent}] {step} -> {status}")

    result = await run_research_agent(
        iso_model=iso_model,
        session_id=session_id,
        broadcast=dummy_broadcast,
        domain_context=iso_model.meta.standard,
    )
    return result

research_result = asyncio.run(test_research_only())
print("Records:", len(research_result.records))
print("Exec summary:", research_result.executive_summary)

In [ ]:
# OPTIONAL: Full pipeline via Super Agent (PDF -> ISO model -> Research)
# NOTE: You do NOT need this if you are testing with the existing JSON model only.
# Leave everything in this cell commented until you actually have a real PDF file.

# from pathlib import Path
# from agents.super_agent import run_pipeline
# from state.session_store import session_store

# pdf_path = Path("data/your_iso_pdf.pdf")  # Replace with real PDF path when you have one

# async def test_full_pipeline(pdf_path: Path, domain_context: str = ""):
#     pdf_bytes = pdf_path.read_bytes()
#     session_id = "notebook-session-pdf"
#
#     await session_store.connect(cfg.redis_url)
#     await session_store.create(session_id, filename=pdf_path.name, domain_context=domain_context)
#
#     async def notebook_broadcast(sid: str, payload: dict):
#         step = payload.get("step", "")
#         agent = payload.get("agent", "")
#         status = payload.get("status", "")
#         if step and agent:
#             print(f"[{agent}] {step} -> {status}")
#
#     state = await run_pipeline(
#         session_id=session_id,
#         pdf_bytes=pdf_bytes,
#         filename=pdf_path.name,
#         broadcast=notebook_broadcast,
#         store=session_store,
#         domain_context=domain_context,
#         skip_research=False,
#     )
#     return state
#
# Example usage once you have a real PDF:
# final_state = asyncio.run(test_full_pipeline(pdf_path, domain_context="nuclear fuel reprocessing"))
# final_state.status, final_state.iso_model is not None, final_state.research_result is not None